In [ ]:
# pip install kagglehub requests pandas pyarrow duckdb unidecode

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
import requests
from unidecode import unidecode

DADOS = Path("..") / "dados"
RAW = DADOS / "raw"
RAW.mkdir(parents=True, exist_ok=True)

URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

In [ ]:
resp = requests.get(URL_IBGE, timeout=60)

resp.raise_for_status

municipios = resp.json()

print(len(municipios))
municipios[0]

In [ ]:
(RAW / "ibge_municipios.json").write_text(
    json.dumps(municipios, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
def uf_de(m):
    # O IBGE devolve a UF em caminhos aninhados diferentes, dependendo do município:
    # (a) regiao-imediata -> regiao-intermediaria -> UF
    # (b) microrregiao -> mesorregiao -> UF
    # "or {}" troca None por um dicionário vazio, para o .get() não quebrar em campo ausente.
    ri = (m.get("regiao-imediata") or {}).get("regiao-intermediaria") or {}
    mi = (m.get("microrregiao") or {}).get("mesorregiao") or {}
    # Usa o caminho (a); se ele não tiver UF, cai no caminho (b); se nenhum tiver, devolve {}.
    return ri.get("UF") or mi.get("UF") or {}

# Lista onde cada município vira um dicionário "plano" (uma linha da futura tabela).
linhas = []
for m in municipios:
    # Descobre o dicionário da UF deste município.
    uf = uf_de(m)
    linhas.append({
        # Código oficial do IBGE
        "ibge_id": m["id"],
        "nome": m["nome"],
        # .get() devolve None (em vez de erro) se o campo não existir.
        "uf_sigla": uf.get("sigla"),
        "uf_nome": uf.get("nome"),
        # A região (Sul, Nordeste...) fica dentro da UF.
        "regiao_sigla": (uf.get("regiao") or {}).get("sigla"),
        "regiao_nome": (uf.get("regiao") or {}).get("nome"),
        # Divisões regionais do IBGE, útil para agregações geográficas.
        "microrregiao": (m.get("microrregiao") or {}).get("nome"),
        "mesorregiao": ((m.get("microrregiao") or {}).get("mesorregiao") or {}).get("nome"),
        "regiao_imediata": (m.get("regiao-imediata") or {}).get("nome"),
        "regiao_intermediaria": ((m.get("regiao-imediata") or {}).get("regiao-intermediaria") or {}).get("nome"),
    })

# Transforma a lista de dicionários em um DataFrame (tabela com uma linha por município).
df = pd.DataFrame(linhas)

# Nome padronizado: sem acento (unidecode), minúsculo (lower) e sem espaços nas pontas (strip).
# .map aplica a função a cada valor da coluna "nome".
df["nome_norm"] = df["nome"].map(lambda s: unidecode(s).lower().strip())

# Registra quando a extração aconteceu (em UTC), para rastreabilidade.
df["extraido_em"] = datetime.now(timezone.utc)

# Validações rápidas:
# shape = (linhas, colunas); is_unique confere que não há código IBGE repetido.
print(df.shape, df["ibge_id"].is_unique)

# Conta valores nulos por coluna: ajuda a achar falhas no achatamento (ex.: UF vazia).
df.isna().sum()

In [ ]:
# Parquet: formato colunar compacto que preserva os tipos e é lido rápido por pandas e DuckDB.
# index=False evita gravar o índice numérico do pandas como uma coluna inútil.
df.to_parquet(DADOS / "ibge_municipios.parquet", index=False)

# Abre (ou cria) o arquivo do banco DuckDB. str() porque a função espera texto, não Path.
con = duckdb.connect(str(DADOS / "warehouse.duckdb"))

# Cria a schema "bronze" (camada de dado bruto) só se ainda não existir.
con.execute("create schema if not exists bronze")

# Cria a tabela a partir do DataFrame "df". O DuckDB reconhece variáveis do Python pelo nome
# dentro do SQL. "create or replace" recria a tabela a cada execução, então rodar de novo
# não duplica linhas.
con.execute("create or replace table bronze.ibge_municipios as select * from df")

# Conferência: conta municípios por UF; devem aparecer 27 UFs. .df() devolve o resultado
# como DataFrame, que o notebook exibe como tabela.
con.execute("select uf_sigla, count(*) n from bronze.ibge_municipios group by 1 order by 1").df()

# Fecha a conexão para liberar o arquivo; o DuckDB aceita só um escritor por vez,
# então deixar aberta pode bloquear outros processos (dbt, Metabase).
con.close()

In [ ]:
# "with" fecha a conexão sozinho ao sair do bloco, então não precisa de con.close().
# read_only=True: aqui só consultamos, não escrevemos.
with duckdb.connect(str(DADOS / "warehouse.duckdb"), read_only=True) as con:
    # Total de municípios, quantas UFs distintas e quantos ficaram sem UF.
    resumo = con.execute("""
        select count(*) as municipios,
               count(distinct uf_sigla) as ufs,
               count(*) filter (where uf_sigla is null) as sem_uf
        from bronze.ibge_municipios
    """).df()
    
    df_ibge = con.execute("""
        select *
        from bronze.ibge_municipios
    """).df()


display(resumo)
display(df_ibge)

In [ ]:
# Endereço "raw" do arquivo no GitHub (conteúdo puro, sem a página do site).
URL_COORDS = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/json/municipios.json"

# Onde o arquivo cru será guardado (mesma pasta "raw" do JSON do IBGE).
destino = RAW / "coordenadas_municipios.json"

# Só baixa se ainda não existir: reexecutar o notebook não faz nova requisição.
if not destino.exists():
    r = requests.get(URL_COORDS, timeout=60)
    # Se o caminho estiver errado (404) ou o GitHub recusar, para aqui com uma mensagem clara.
    r.raise_for_status()
    # r.content são os bytes originais; gravar bytes evita problemas de codificação.
    destino.write_bytes(r.content)

# Lê o arquivo salvo e converte para lista de dicionários, como antes.
coords_raw = json.loads(destino.read_text(encoding="utf-8-sig"))
print(len(coords_raw)) 



In [ ]:
coords = pd.DataFrame(coords_raw)

# AJUSTE à esquerda dos ":"
# Padronizado para ibge_id / lat / lng.
coords = coords.rename(columns={"codigo_ibge": "ibge_id", "latitude": "lat", "longitude": "lng", "fuso_horario": "fuso_horario"})

# Mantém só o necessário (copy() evita avisos de modificação de "fatia").
coords = coords[["ibge_id", "lat", "lng", "fuso_horario"]].copy()

# Garante tipos corretos: código inteiro (igual ao da tabela do IBGE) e coordenadas numéricas.
coords["ibge_id"] = coords["ibge_id"].astype(int)
coords["lat"] = pd.to_numeric(coords["lat"], errors="coerce")
coords["lng"] = pd.to_numeric(coords["lng"], errors="coerce")
coords["fuso_horario"] = coords["fuso_horario"].astype(str).str.strip()

# Checagens: códigos repetidos e coordenadas fora do Brasil (aprox. lat -34 a 6, lng -74 a -32).
print("ids repetidos:", coords["ibge_id"].duplicated().sum())
print("fora do Brasil:", (~coords["lat"].between(-34, 6) | ~coords["lng"].between(-74, -32)).sum())



In [ ]:
# Base de municípios salva antes. Remove lat/lng se já existirem (permite reexecutar sem criar colunas duplicadas).
base = pd.read_parquet(DADOS / "ibge_municipios.parquet").drop(columns=["lat", "lng", "fuso_horario"], errors="ignore")

# Left join: mantém todos os municípios do IBGE; indicator marca quem achou coordenada.
teste = base.merge(coords, on="ibge_id", how="left", indicator=True)
print(teste["_merge"].value_counts())

# Municípios sem coordenada
teste.loc[teste["lat"].isna(), ["ibge_id", "nome", "uf_sigla"]]

In [ ]:
# validate="one_to_one" falha se houver código repetido em qualquer lado, evitando linhas duplicadas.
final = base.merge(coords, on="ibge_id", how="left", validate="one_to_one")

# Sobrescreve o Parquet e recria a tabela no DuckDB já com lat e lng.
final.to_parquet(DADOS / "ibge_municipios.parquet", index=False)

with duckdb.connect(str(DADOS / "warehouse.duckdb")) as con:
    con.execute("create or replace table bronze.ibge_municipios as select * from final")
    # fetchall() devolve uma lista simples (evita o aviso do pandas do .df()).
    print(con.execute("select count(*) as total, count(lat) as com_coord from bronze.ibge_municipios").fetchall())